CD704: Exercises, Week 7

17 November 2026

You may need LLM / AI help for the following exercises.


# Exercise 1 (Kernel Ridge Regression). 

Load the $\mathtt{scikit-learn}$ California Housing dataset and select a random subset of size 2000. 
Split this subset into training (80%) and test (20%) sets.

1. Fit a $\mathtt{KernelRidge}$ model with an RBF kernel and default hyperparameters directly
on the unscaled training data, and record the test MSE and $R^2$. Next, build a
pipeline combining $\mathtt{StandardScaler}$ and $\mathtt{KernelRidge}$, evaluate it on the same
split, and compare the results.

2. Using pipelines with $\mathtt{StandardScaler}$, try $\mathtt{KernelRidge}$ with four kernels: $\mathtt{linear}$,
$\mathtt{poly}$, $\mathtt{rbf}$, and $\mathtt{laplacian}$. Use $\mathtt{GridSearchCV}$ or $\mathtt{RandomizedSearchCV}$ to tune the
regularisation parameter alpha along with the kernel parameters ($\mathtt{gamma}$, $\mathtt{degree}$,
$\mathtt{coef0}$. Compare the best test $R^2$ across the four kernels. How does the linear
kernel compare to standard ridge regression?

3. Fix a test set of 1000 instances from the full California Housing dataset. Using your
best tuned RBF pipeline, measure both the fitting time ($\mathtt{.fit()}$) and the prediction
time ($\mathtt{.predict()}$) for training set sizes $N$ growing from 200 to 8000 (for example,
200, 500, 1000, 2000, 4000, 8000). Plot both running times against $N$ (you may need
log–log axes). Do your empirical timings relate to the $O(n^3)$ training complexity in
the lecture?

## Response

### Download and Split the Dataset

In [30]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import PAIRWISE_KERNEL_FUNCTIONS, pairwise_kernels
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load the california housing dataset and split it into training and test sets.

california = fetch_california_housing(data_home="../scikit_learn_data", download_if_missing=False)
X = california.data
y = california.target

df = pd.DataFrame(X, columns=california.feature_names)
df = pd.concat([df, pd.DataFrame(y, columns=['target'])], axis=1)

# Select 200 random samples from the dataset
df_sampled = df.sample(n=200, random_state=42)

# Split the sampled dataset into training and test sets
X_sampled = df_sampled.drop(columns=['target'])
y_sampled = df_sampled['target']
X_train, X_test, y_train, y_test = train_test_split(X_sampled, y_sampled, test_size=0.2, random_state=42)

# Convert to numpy arrays
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)


### Part 1. KernelRidge with and without Scaling

#### WITHOUT Scaling

In [31]:
kernelRidge = KernelRidge(alpha=1.0, kernel='rbf', gamma=0.1)
kernelRidge.fit(X_train, y_train)
mse = mean_squared_error(y_test, kernelRidge.predict(X_test))
r2 = kernelRidge.score(X_test, y_test)
print(f"UNSCALED Kernel Ridge MSE: {mse}, R2: {r2}")

UNSCALED Kernel Ridge MSE: 4.585026534705182, R2: -2.94638862454016


#### WITH Scaling

In [32]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kernel_ridge', KernelRidge(alpha=1.0, kernel='rbf', gamma=0.1))
])

pipeline.fit(X_train, y_train)
mse = mean_squared_error(y_test, pipeline.predict(X_test))
r2 = pipeline.score(X_test, y_test)
print(f"SCALED Kernel Ridge MSE: {mse}, R2: {r2}")

SCALED Kernel Ridge MSE: 0.4812939212753811, R2: 0.5857444135590764


#### Comparing the Results

Text here.

### Part 2. Survey Kernels and Search Methods

In [33]:
kernelRidge = KernelRidge(alpha=1.0, kernel='rbf', gamma=0.1)
print(f"params: {kernelRidge.get_params()}")

params: {'alpha': 1.0, 'coef0': 1, 'degree': 3, 'gamma': 0.1, 'kernel': 'rbf', 'kernel_params': None}


In [41]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kr', KernelRidge())
])

my_param_grid = {
    'kr__alpha': np.logspace(-5, 3, 5),
    'kr__coef0': [0.0, 1.0, 10.0, 100.0],
    'kr__degree': [2, 3, 4, 5],
    'kr__gamma': np.logspace(-5, 5, 5),
    'kr__kernel': ['linear', 'poly', 'rbf', 'laplacian']
}

my_gridsearch = GridSearchCV(pipeline, param_grid=my_param_grid, cv=5)
my_gridsearch.fit(X_train, y_train)
print(f"Best model score: {my_gridsearch.best_score_}")
print(f"Best model parameters: {my_gridsearch.best_params_}")
print(f"Best model: {my_gridsearch.best_estimator_}")
best_model = my_gridsearch.best_estimator_



/Users/swerdna/Library/CloudStorage/GoogleDrive-swerdna.neb@gmail.com/My Drive/Brighton/University of Brighton/MSc Data Analytics/CD704 Machine Learning/CD704-ML/.venv/lib/python3.15/site-packages/sklearn/linear_model/_ridge.py:266: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/swerdna/Library/CloudStorage/GoogleDrive-swerdna.neb@gmail.com/My Drive/Brighton/University of Brighton/MSc Data Analytics/CD704 Machine Learning/CD704-ML/.venv/lib/python3.15/site-packages/sklearn/linear_model/_ridge.py:266: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/swerdna/Library/CloudStorage/GoogleDrive-swerdna.neb@gmail.com/My Drive/Brighton/University of Brighton/MSc Data Analytics/CD704 Machine Learning/CD704-ML/.venv/lib/python3.15/site-packages/sklearn/linear_model/_ridge.py:266: UserWarning: Singular matrix in solving dual problem. Using least-squares solution 

Best model score: 0.5932863533441439
Best model parameters: {'kr__alpha': np.float64(0.001), 'kr__coef0': 10.0, 'kr__degree': 5, 'kr__gamma': np.float64(1e-05), 'kr__kernel': 'poly'}
Best model: Pipeline(steps=[('scaler', StandardScaler()),
                ('kr',
                 KernelRidge(alpha=np.float64(0.001), coef0=10.0, degree=5,
                             gamma=np.float64(1e-05), kernel='poly'))])


In [38]:
np.logspace(-3, 3, 7)


array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])


# Exercise 2 (Kernel Support Vector Classification). 

Load the scikit-learn Breast Cancer Wisconsin dataset and split it into training (80%) and test (20%) sets using a stratified
split.
1. Inspect the ranges of the features in the dataset. Fit a support vector classifier SVC(kernel="rbf") with default parameters on the unscaled training data
and evaluate its test accuracy (and inspect the predicted classes). Then wrap
StandardScaler and SVC(kernel="rbf") inside a Pipeline, re-evaluate and compare the results.

2. Using pipelines with StandardScaler, try SVC with kernel="linear", kernel="poly",
and kernel="rbf". Tune the regularisation parameter and kernel parameters via
cross-validation. Compare the test accuracies and the number of support vectors
retained by each winning model.




# Exercise 3 (Kernels and Explainability). 

Load the scikit-learn Breast Cancer Wisconsin dataset and split it into training (80%) and test (20%) sets using a stratified
split.

1. Build pipelines with $\mathtt{StandardScaler}$ for a linear SVM and an SVM with an RBF
kernel. Compare their test accuracies. For the linear model, extract and plot the
top 10 features with the largest absolute weights $|w_j|$. Note that the linear decision
score $\vec w^T \vec x + w_0$ decomposes directly into exact feature contributions $w_j x_j$
for any
standardised input $\vec x$.

2. Using the $\mathtt{lime}$ package, generate local explanations for three test instances: a
confident benign prediction, a confident malignant prediction, and a borderline or
misclassified instance. Do this for both the linear pipeline and the RBF SVM
pipeline (passing the pipeline’s .predict proba method so features remain in their
original units). Compare LIME’s feature weights for the linear model with the exact
contributions wjxj, and check how much LIME explanations vary if you re-run the
explainer with a different random seed.

3. Using the shap package, compute SHAP values for both pipelines (for example, using shap.Explainer on the linear model and shap.KernelExplainer with a summarised background set such as shap.kmeans(X train, 25) for the RBF SVM).
Produce a SHAP summary plot across the test set and waterfall plots for the same
three instances examined before. Compare the global feature importance ranking
of the RBF SVM with the linear model weights above.